In [1]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

import galsim
import numpy as np
import anacal
import os
import matplotlib.pyplot as plt
import fitsio

In [2]:
def simulate_exp_gal_stamp(
    hlr_arcsec: float,
    g1: float = 0.02,
    g2: float = 0.0,
    stamp_size: int = 64,
    pixel_scale: float = 0.2, 
    psf_beta: float = 2.0,
    psf_fwhm_arcsec: float = 0.8,
):
    # --- Profiles ---
    gal = galsim.Exponential(half_light_radius=hlr_arcsec, flux=200).shear(g1=g1, g2=g2)
    psf = galsim.Moffat(beta=psf_beta, fwhm=psf_fwhm_arcsec)

    final = galsim.Convolve([gal, psf])

    gal_img = galsim.ImageF(stamp_size, stamp_size, scale=pixel_scale)
    psf_img = galsim.ImageF(stamp_size, stamp_size, scale=pixel_scale)
    gal_img.setOrigin(0, 0)
    psf_img.setOrigin(0, 0)

    half_pixel_offset = galsim.PositionD(0.5, 0.5)

    final.drawImage(image=gal_img, method="fft", offset=half_pixel_offset)
    psf.drawImage(image=psf_img, method="fft", offset=half_pixel_offset)

    return gal_img, psf_img


def measure(
    gal_array: np.ndarray,
    psf_array: np.ndarray,
    pixel_scale: float,
    noise_variance: float = 0.25,
    noise_array: np.ndarray | None = None,
    g1_input: float = 0.02,
):
    npix = gal_array.shape[0]
    assert gal_array.shape == (npix, npix)
    assert psf_array.shape == (npix, npix)

    dtype = np.dtype([("y", np.int32), ("x", np.int32)])
    detection = np.empty(1, dtype=dtype)
    detection["y"] = npix // 2
    detection["x"] = npix // 2

    # FPFS config (same as your example)
    fpfs_config = anacal.fpfs.FpfsConfig(
        sigma_shapelets=0.50,
        sigma_shapelets1=0.50,
    )

    catalog = anacal.fpfs.process_image(
        fpfs_config=fpfs_config,
        mag_zero=30.0,
        gal_array=gal_array,
        psf_array=psf_array,
        pixel_scale=pixel_scale,
        noise_variance=max(noise_variance, 0.23),
        noise_array=noise_array,
        detection=detection,
        return_only_linear_modes=True,
        pack_linear_modes=True,
    )
    return catalog

In [3]:
noise_variance = 0.23
pixel_scale = 0.2
g1_input = 0.02

hlr = 0.4
gal_img, psf_img = simulate_exp_gal_stamp(hlr_arcsec=hlr, pixel_scale=pixel_scale)
gal_array = gal_img.array
psf_array = psf_img.array

catalog = measure(
    gal_array=gal_array,
    psf_array=psf_array,
    pixel_scale=pixel_scale,
    noise_variance=noise_variance,
    noise_array=None,
    g1_input=g1_input,
)

In [4]:
catalog.dtype.names

('y',
 'x',
 'fpfs_m00',
 'fpfs_m20',
 'fpfs_m22c',
 'fpfs_m22s',
 'fpfs_m40',
 'fpfs_m42c',
 'fpfs_m42s',
 'fpfs_m44c',
 'fpfs_m44s',
 'fpfs_m60',
 'fpfs_m64c',
 'fpfs_m64s',
 'fpfs_v0',
 'fpfs_v1',
 'fpfs_v2',
 'fpfs_v3',
 'fpfs_v0r1',
 'fpfs_v1r1',
 'fpfs_v2r1',
 'fpfs_v3r1',
 'fpfs_v0r2',
 'fpfs_v1r2',
 'fpfs_v2r2',
 'fpfs_v3r2',
 'fpfs_n00',
 'fpfs_n20',
 'fpfs_n22c',
 'fpfs_n22s',
 'fpfs_n40',
 'fpfs_n42c',
 'fpfs_n42s',
 'fpfs_n44c',
 'fpfs_n44s',
 'fpfs_n60',
 'fpfs_n64c',
 'fpfs_n64s',
 'fpfs_u0',
 'fpfs_u1',
 'fpfs_u2',
 'fpfs_u3',
 'fpfs_u0r1',
 'fpfs_u1r1',
 'fpfs_u2r1',
 'fpfs_u3r1',
 'fpfs_u0r2',
 'fpfs_u1r2',
 'fpfs_u2r2',
 'fpfs_u3r2',
 'fpfs1_m00',
 'fpfs1_m20',
 'fpfs1_m22c',
 'fpfs1_m22s',
 'fpfs1_m40',
 'fpfs1_m42c',
 'fpfs1_m42s',
 'fpfs1_m44c',
 'fpfs1_m44s',
 'fpfs1_m60',
 'fpfs1_m64c',
 'fpfs1_m64s',
 'fpfs1_n00',
 'fpfs1_n20',
 'fpfs1_n22c',
 'fpfs1_n22s',
 'fpfs1_n40',
 'fpfs1_n42c',
 'fpfs1_n42s',
 'fpfs1_n44c',
 'fpfs1_n44s',
 'fpfs1_n60',
 'fpfs1_n64c',
 'f

In [8]:
catalog["fpfs1_n40"]

array([0.])